# 04 · MTrans inference on KITTI-360 (the object teacher's boxes)

Runs the trained MTrans checkpoint (notebook 03, the **50-epoch teacher of record**) on the 2D detector's boxes for each KITTI-360 drive (notebook 02): every detection plus the LiDAR points in its frustum gives a 3D box in the LiDAR frame with its raw confidence (`conf_iou_raw`, regressed to the 3D IoU, never rescaled). Detections with no frustum point keep a row (`too_few_points`, no 3D box): a detected object must never become free road downstream.

Output: `objects_v1/kitti360/<sequence>.jsonl` (one line per detection) and `<sequence>.manifest.json` (checkpoint and detections hashes). The student reads these files for its object channels, box occluders and occupancy pseudo-labels.

Optional: the same on KITTI val with detector boxes (`RUN_KITTI_VAL`), for the teacher's quality with real detections (gate G4).

## Before you run

1. **Settings → Accelerator:** GPU T4. **Internet:** On (code clone).
2. **Add Input:** the MTrans training output (`03-mtrans-teacher`, folder `mtrans_v1` with `ckpt/last.pt`); the detector output holding `kitti360/<sequence>.txt` (notebook 02 run on these drives); each drive's data-prep output and the KITTI-360 starter (calibration; images for drive 0000). For `RUN_KITTI_VAL`: also `garymk/kitti-3d-object-detection-dataset` and `ocgf-detector2d-v1`.
3. **Save Version → Save & Run All.**

In [ ]:
from pathlib import Path

REPO_URL  = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF   = "p0/skeleton-geometry"       # branch, tag or commit (main once merged); pin a commit for the final run
MTRANS_EXPERIMENT = "mtrans_v1"          # notebook 03's output folder
DRIVES    = ["2013_05_28_drive_0000_sync"]
RUN_KITTI_VAL = False                    # also KITTI val with detector boxes (teacher quality, G4)
MIN_SCORE = 0.1                          # 2D score floor: recall-oriented, scores flow on as soft inputs
MIN_POINTS = 1                           # frustum points needed for a 3D box
BATCH_SIZE = 32
THREADS   = 32                           # parallel readers: /kaggle/input is slow per file
INPUT_ROOT = Path("/kaggle/input")
OUT_DIR   = Path("/kaggle/working/objects_v1")   # kept as the notebook output
CACHE     = Path("/kaggle/temp/mtrans_cache")    # per-frame point caches
STAGE_DIR = Path("/kaggle/temp/kitti")
CODE_DIR  = Path("/kaggle/temp/OCGF")

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd, cwd=None):
    # run a command, stream its output into this notebook, and stop Run All if it fails
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

CODE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))
for name in [m for m in sys.modules if m == "ocgf" or m.startswith("ocgf.")]:
    del sys.modules[name]

from ocgf.utils.provenance import environment
print(environment())
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

## Inputs

The MTrans checkpoint (it must be finished: the learning rate annealed), each drive's detections and data.

In [ ]:
import json
from ocgf.datasets.locate import describe_prepared, find_detections, find_runs

runs = find_runs(INPUT_ROOT, MTRANS_EXPERIMENT)
assert runs, f"no {MTRANS_EXPERIMENT}/ckpt/last.pt among the inputs: attach the MTrans training output"
CKPT = runs[0] / "ckpt" / "last.pt"
progress = json.loads((CKPT.parent / "progress.json").read_text())
print(f"MTrans checkpoint {CKPT}: {progress['epoch'] + 1} of {progress['epochs']} epochs")
assert progress["epoch"] + 1 >= progress["epochs"], (
    "this checkpoint is not finished (its learning rate has not annealed): only the finished teacher of record may "
    "produce object files")

DETECTIONS = {}
for seq in DRIVES:
    det = find_detections(INPUT_ROOT, seq)
    if det is None:
        print(f"{seq}: skipped, no kitti360/{seq}.txt (run notebook 02 on this drive and attach its output)")
        continue
    print(describe_prepared(INPUT_ROOT, seq))
    n = sum(1 for _ in open(det))
    print(f"{seq}: detections {det} ({n} boxes)")
    DETECTIONS[seq] = det
assert DETECTIONS or RUN_KITTI_VAL, "nothing to run on"


## Run

One drive at a time: the frames' points are cached locally (parallel reads), then every detection gets its row.

In [ ]:
for seq, det in DETECTIONS.items():
    sh(f'{sys.executable} scripts/run_mtrans.py --checkpoint "{CKPT}" --input-root "{INPUT_ROOT}" --sequence {seq} '
       f'--detections "{det}" --out "{OUT_DIR / "kitti360" / (seq + ".jsonl")}" --min-score {MIN_SCORE} '
       f'--min-points {MIN_POINTS} --cache "{CACHE / seq}" --batch-size {BATCH_SIZE} --threads {THREADS}', cwd=CODE_DIR)

if RUN_KITTI_VAL:
    kitti_det = next(INPUT_ROOT.rglob("detector2d_v1/kitti/training.txt"), None)
    assert kitti_det, "attach ocgf-detector2d-v1 (detector2d_v1/kitti/training.txt) for KITTI val"
    sh(f'{sys.executable} scripts/run_mtrans.py --checkpoint "{CKPT}" --input-root "{INPUT_ROOT}" --kitti-split val '
       f'--detections "{kitti_det}" --out "{OUT_DIR / "kitti" / "val.jsonl"}" --min-score {MIN_SCORE} '
       f'--min-points {MIN_POINTS} --cache "{CACHE / "kitti"}" --stage-dir "{STAGE_DIR}" --batch-size {BATCH_SIZE} '
       f'--threads {THREADS}', cwd=CODE_DIR)

## Summary

Per drive: detections, how many got a 3D box, and per class the count and mean confidence.

In [ ]:
from collections import Counter, defaultdict

for path in sorted(OUT_DIR.rglob("*.jsonl")):
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    status = Counter(r["status"] for r in rows)
    frames = len({r["fid"] for r in rows})
    print(f"{path.relative_to(OUT_DIR)}: {len(rows)} detections in {frames} frames; {dict(status)}")
    conf = defaultdict(list)
    for r in rows:
        if r["conf_iou_raw"] is not None:
            conf[r["cls"]].append(r["conf_iou_raw"])
    for cls, c in sorted(conf.items()):
        print(f"  {cls:14s} {len(c):6d} boxes, mean conf_iou_raw {sum(c) / len(c):.3f}")
sh(f'du -sh "{OUT_DIR}"/*')

## After the run

Save the output as the private dataset **`ocgf-mtrans-objects-v1`** (Output → New Dataset). Notebooks 07 and 08 attach it for the object experiments (E1, E2). Objects come from the 50-epoch teacher of record only; teachers are never mixed.